## Task 2: simple baseline model

Import all the dependencies en setup the Keras backend. You select the backend you want to use and change os.environ based on the availability of a GPU to accelerate computation. 

In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import os
# Set Keras backend (can be 'tensorflow', 'torch', 'jax')
os.environ["KERAS_BACKEND"] = "torch"

# CPU only
# os.environ["CUDA_VISIBLE_DEVICES"] = ""

# Specific GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
import keras
from keras import layers, optimizers, losses

In [4]:
print(f"Keras backend: {keras.backend.backend()}")

Keras backend: torch


Load the data from the preprocessing notebook

In [5]:
X_train = np.load("X_train.npy")
y_train = np.load("y_train.npy")
X_val = np.load("X_val.npy")
y_val = np.load("y_val.npy")
X_test = np.load("X_test.npy")
y_test = np.load("y_test.npy")


### Step 1: 

Set up classification and choose metrics to eveluate the classifier. Because the model will be used in a medical context, the metrics
will be adapted to this. We will use:
 - Accuracy
 - Recall
 - Specifity
 - F1 score
 - ROC-AUC

### Step 2: 
Build the initial model. The architecture is as follows:

| Layer | Output shape | Description |
|---|---|---|
| Input | 128 × 128 × 1 | Grayscale X-ray image |
| Conv2D(32, 3×3, same) + BN + ReLU | 128 × 128 × 32 | Detect low-level features (edges, gradients) |
| MaxPooling2D(2×2) | 64 × 64 × 32 | Downsample, retain strongest activations |
| Conv2D(64, 3×3, same) + BN + ReLU | 64 × 64 × 64 | Detect mid-level features (textures, shapes) |
| MaxPooling2D(2×2) | 32 × 32 × 64 | Downsample again |
| Conv2D(128, 3×3, same) + BN + ReLU | 32 × 32 × 128 | Detect high-level features (structures, patterns) |
| MaxPooling2D(2×2) | 16 × 16 × 128 | Final spatial compression |
| Flatten | 32768 | Reshape feature maps to 1D vector |
| Dense(256) + ReLU | 256 | Learn combinations of extracted features |
| Dropout(0.5) | 256 | Regularisation, prevent overfitting |
| Dense(1) + Sigmoid | 1 | Output P(disease) for binary classification |

In [6]:
model = keras.Sequential(
    [
        layers.Input(shape=(128, 128, 1)),
        # Block 1: Convolution + Pooling
        layers.Conv2D(32, kernel_size=(3, 3), padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        # Block 2: Convolution + Pooling
        layers.Conv2D(64, kernel_size=(3, 3), padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        # Block 3: Convolution + Pooling
        layers.Conv2D(128, kernel_size=(3, 3), padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        # Flatten
        layers.GlobalAveragePooling2D(),
        # Classification head
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(1, activation="sigmoid"),
    ]
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 110,209 (430.50 KB)

 Trainable params: 109,761 (428.75 KB)

 Non-trainable params: 448 (1.75 KB)

## Step 3: train the model

The training will be done using the **binairy cross entropy** loss function. This function penalyses confident wrong mispredictions. If the model predicts P(desease) = 0.95 and the true label is 0, the log term produceses a large loss. When the model is confidently correct, the loss approaches 0. It pairs good with the sigmoid activation of the last layer in the network. Because this layer squashes the output to the [0,1] interval, the ouput can be interpreted as a probability and in turn can be used in the cross entropy function.

For model evaluation the metrics mentioned in step 1 will be used.

## 

In [7]:
loss = losses.BinaryCrossentropy()

In [8]:
optimizer = optimizers.Adam(learning_rate = 0.001)

In [9]:
model.compile(optimizer=optimizer, loss=loss, metrics=["accuracy"])

In [10]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=y_train
)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}
print(class_weight_dict)  # should be something like {0: 0.75, 1: 1.5}

{0: np.float64(0.75), 1: np.float64(1.5)}


In [10]:
history = model.fit(
    X_train,
    y_train,
    batch_size=16,
    epochs=50,
    validation_data=(X_val, y_val),
)

Epoch 1/50


94/94 ━━━━━━━━━━━━━━━━━━━━ 5s 47ms/step - accuracy: 0.6413 - loss: 0.6602 - val_accuracy: 0.6226 - val_loss: 0.6714
Epoch 2/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 4s 43ms/step - accuracy: 0.6600 - loss: 0.6461 - val_accuracy: 0.6038 - val_loss: 0.6661
Epoch 3/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 9s 93ms/step - accuracy: 0.6653 - loss: 0.6352 - val_accuracy: 0.6164 - val_loss: 0.6645
Epoch 4/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 9s 92ms/step - accuracy: 0.6700 - loss: 0.6353 - val_accuracy: 0.6226 - val_loss: 0.6877
Epoch 5/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 9s 91ms/step - accuracy: 0.6727 - loss: 0.6384 - val_accuracy: 0.6038 - val_loss: 0.6717
Epoch 6/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 8s 91ms/step - accuracy: 0.6740 - loss: 0.6348 - val_accuracy: 0.6164 - val_loss: 0.6655
Epoch 7/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 9s 98ms/step - accuracy: 0.6760 - loss: 0.6332 - val_accuracy: 0.6164 - val_loss: 0.6596
Epoch 8/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 11s 113ms/step - accuracy: 0.6660 - loss: 0.6394 - val_accuracy: 0.6226 - val_loss: 

In [ ]:
def plot_training_history(history, ax=None, title=None):
    if ax is None:
        _, ax = plt.subplots()
    ax.plot(history.history["accuracy"], label="Train Accuracy")
    ax.plot(history.history["val_accuracy"], label="Validation Accuracy")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.set_title(title)
    ax.legend(loc="lower right")
    return ax